# Ordered Logistic Regression Results (FAIR^2) Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR² dataset containing ordered logistic regression results for adoption predictors of indigenous and modern knowledge in rangeland management practices in Northern Kenya.

### Dataset Source
The dataset is defined via a Croissant schema URL and leverages the `mlcroissant` library for programmatic access.

In [ ]:
# Install the `mlcroissant` library. Uncomment if not already installed in your environment.
!pip install mlcroissant --quiet

## 1. Data Loading
Load dataset metadata and prepare the Croissant dataset object using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Set the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata # metadata is an object with attributes

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Identifier: {getattr(metadata, 'identifier', None)}")
print(f"Authors: {getattr(metadata, 'author', None)}")
print(f"Temporal Coverage: {getattr(metadata, 'temporalCoverage', None)}")

## 2. Data Overview
Review available record sets, their fields, and corresponding `@id` values using the Croissant API.

> **Note:** All Croissant entities (record sets, fields, columns) are referenced using their `@id`.

In [ ]:
# List all available record sets and their details
record_sets = list(dataset.record_sets)
if not record_sets:
    print("No record sets defined in the schema.")
else:
    for rs in record_sets:
        print(f"Record set @id: {rs.id}")
        print(f"  Name: {rs.name if hasattr(rs, 'name') else ''}")
        print(f"  Description: {rs.description if hasattr(rs, 'description') else ''}")
        if hasattr(rs, 'fields'):
            print('  Fields:')
            for f in rs.fields:
                print(f"    - Field @id: {f.id}, Name: {getattr(f, 'name', '')}, DataType: {getattr(f, 'data_type', '')}")
        print("")

## 3. Data Extraction
Extract one or more record sets from the Croissant dataset. If multiple record sets are present, each will be loaded into a pandas DataFrame for analysis. All field IDs used are `@id` values.

In [ ]:
# Prepare list of record set @id values for extraction
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if not records:
        print(f"No records found in record set {rs_id}")
        continue
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded dataframe for record set {rs_id} with shape {df.shape}")
    print(f"Columns: {df.columns.tolist()}")

# Example: display the first few records of the first record set if available
if dataframes:
    first_rs_id = record_set_ids[0]
    print(f"\nPreview of the first record set ({first_rs_id}):")
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's apply some common data processing steps:
- Filter for records based on a numeric field
- Normalize a numerical column
- Group data based on a categorical field

> Use `@id` to refer to your columns. You may adjust the field IDs below based on the DataFrame columns discovered earlier. If unsure, inspect the columns printed above.

In [ ]:
# For demonstration, select a record set and field(s) by their @id
if dataframes:
    # Choose record set to analyze
    record_set_id = list(dataframes.keys())[0]  # Change if more than one
    df = dataframes[record_set_id]
    print(f"Analyzing record set: {record_set_id}")

    # Display columns to guide field selection
    print(df.columns.tolist())

    # Try to infer a numeric field (commonly named or identified field)
    numeric_fields = df.select_dtypes(include=['number']).columns.tolist()
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
    else:
        # Fallback: try to find a numeric-looking field (by typical names)
        numeric_field_candidates = [col for col in df.columns if 'log' in col.lower() or 'value' in col.lower() or 'coefficient' in col.lower() or 'likelihood' in col.lower()]
        numeric_field_id = numeric_field_candidates[0] if numeric_field_candidates else None

    if numeric_field_id is not None:
        print(f"Using numeric field @id: {numeric_field_id}")

        threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold:.3f}:")
        display(filtered_df.head())

        # Normalizing the field
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized column '{numeric_field_id}' added:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Try grouping by a common categorical/id field
        group_field_candidates = [col for col in df.columns if col != numeric_field_id and (df[col].dtype == object or df[col].dtype.name == 'category')]
        if group_field_candidates:
            group_field = group_field_candidates[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
            print(f"Grouped means of '{numeric_field_id}' by '{group_field}':")
            display(grouped_df.head())
        else:
            print('No suitable categorical field found for grouping.')
    else:
        print('No quantitative/numeric field found for EDA.')
else:
    print('No dataframes loaded to perform EDA.')

## 5. Visualization
Let's visualize a column distribution or a correlation.
> Adjust the `numeric_field_id` and grouping fields below as appropriate for your dataset. You may refer to column names printed in previous steps.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field_id is not None:
    # Histogram of the numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20, color='skyblue')
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()

    # Boxplot by group field (if available)
    if 'group_field' in locals() and group_field in df.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No field available for visualization.')

## 6. Conclusion
- This notebook demonstrated loading and exploring a Croissant-structured dataset using the `mlcroissant` library.
- All dataset entities were referenced using their `@id` per best practices.
- You can extend the above workflow with further analysis, model training, or exporting summaries as needed.